# Demo 5 — Carrying the overhead: window & cost
<a id="top"></a>

```yaml
title: Demo 5 — Carrying the overhead: window & cost
keywords: context window, cost, pricing, input output asymmetry, staircase, budgeting
estimated duration: 15
```

> **Lesson L01**, teacher demo 5 — the payoff. Written lecture [L0102_lecture.md](L0102_lecture.md) §7–8.
>
> Fully **offline**: token counts via `tiktoken`, costs via **illustrative** rates. Because you
> re-send the preamble + history every call (no memory), those tokens hit two budgets at once —
> **space** (the window) and **money** (cost).

**Contents**

1. [The window is one shared budget](#1-the-window-is-one-shared-budget)
2. [Three failure modes](#2-three-failure-modes)
3. [Cost: per token, both directions](#3-cost-per-token-both-directions)
4. [The staircase and orders of magnitude](#4-the-staircase-and-orders-of-magnitude)
5. [Common pitfalls: L01's four gotchas](#5-common-pitfalls-l01s-four-gotchas)

In [1]:
from __future__ import annotations

import tiktoken

ENC = tiktoken.get_encoding("cl100k_base")
WINDOW = 200_000  # Claude Sonnet 4.6 standard context window (tokens)

# ⚠️ ILLUSTRATIVE rates only ($ per 1,000,000 tokens). Pricing changes — pull current Claude
# Sonnet numbers from Anthropic's pricing page before quoting any real figure.
INPUT_RATE = 3.0
OUTPUT_RATE = 15.0


def window_meter(segments: dict[str, int], total: int = WINDOW, width: int = 40) -> None:
    used = sum(segments.values())
    filled = round(width * min(used, total) / total)
    print(f"[{'#' * filled}{'-' * (width - filled)}] {used:,} / {total:,} tokens")
    for name, n in segments.items():
        print(f"    {name:16s} {n:>8,}")


def call_cost_usd(input_tokens: int, output_tokens: int) -> float:
    return input_tokens / 1e6 * INPUT_RATE + output_tokens / 1e6 * OUTPUT_RATE

## 1. The window is one shared budget

The preamble, every prior turn, the current input, and the reserved output all draw from the
**same** 200k window. Watch it fill as history grows.

In [2]:
window_meter({"preamble": 300, "history": 1_200, "current input": 250, "reserved output": 1_024})
print()
# ...now the same call after a long conversation:
window_meter({"preamble": 300, "history": 160_000, "current input": 250, "reserved output": 1_024})

[#---------------------------------------] 2,774 / 200,000 tokens
    preamble              300
    history             1,200
    current input         250
    reserved output     1,024

[################################--------] 161,574 / 200,000 tokens
    preamble              300
    history           160,000
    current input         250
    reserved output     1,024


**What that `history` segment actually is.** It isn't a magic number — it's the literal
transcript of prior turns, re-sent verbatim on every call (there is no server-side memory). Here
is a small three-turn block, counted the same way and dropped into the meter:

In [3]:
# A history block is just the list of prior turns you carry forward and re-send each call.
HISTORY = [
    ("user", "Draft a release note for the new CSV export."),
    ("assistant", "Added CSV export — download any table as a .csv file."),
    ("user", "Shorter, and say it's on the Reports page."),
    ("assistant", "Added CSV export on the Reports page."),
]
for role, content in HISTORY:
    print(f"{role:>9}: {content}")

history_tokens = sum(len(ENC.encode(content)) for _, content in HISTORY)
print(f"\nthis block = {len(HISTORY)} turns, {history_tokens} tokens — re-sent on the NEXT call\n")
window_meter(
    {"preamble": 300, "history": history_tokens, "current input": 40, "reserved output": 1_024}
)

     user: Draft a release note for the new CSV export.
assistant: Added CSV export — download any table as a .csv file.
     user: Shorter, and say it's on the Reports page.
assistant: Added CSV export on the Reports page.

this block = 4 turns, 43 tokens — re-sent on the NEXT call

[----------------------------------------] 1,407 / 200,000 tokens
    preamble              300
    history                43
    current input          40
    reserved output     1,024


## 2. Three failure modes

| Failure mode        | What happens                                          | Danger |
| ------------------- | ----------------------------------------------------- | ------ |
| Hard rejection      | API refuses the request before running it             | Loud — easy to catch |
| Silent truncation   | Some frameworks quietly drop the oldest turns         | **Quiet — the call succeeds but the model "forgot"** |
| Quality degradation | Even when it fits, mid-context material gets lost      | Subtle — vaguer answers, no error |

The dangerous one is **silent truncation**: no error, but the model no longer has context you
assumed. It motivates context management (L19) and RAG (L21).

[↑ Back to top](#top)

## 3. Cost: per token, both directions

Separate input and output rates, on every call. Output usually costs **3–5× more** — so a long
prompt + short answer often beats a short prompt + long answer.

In [4]:
print(f"2000 in / 50 out : ${call_cost_usd(2000, 50):.5f}")
print(f"  50 in / 2000 out: ${call_cost_usd(50, 2000):.5f}   <- output is the expensive direction")

2000 in / 50 out : $0.00675
  50 in / 2000 out: $0.03015   <- output is the expensive direction


## 4. The staircase and orders of magnitude

No memory means every turn re-sends the whole history **plus the preamble** — so input tokens
climb even when each message is small.

In [5]:
cumulative = 0
session_cost = 0.0
for turn in range(1, 6):
    cumulative += 200  # ~200 new tokens/turn, but ALL prior tokens are re-sent
    cost = call_cost_usd(cumulative, 60)
    session_cost += cost
    print(f"turn {turn}: input re-sent={cumulative:>4} tokens  call=${cost:.5f}")
print(f"\n5-turn session ~ ${session_cost:.4f}")

one_call = call_cost_usd(2000, 500)
print("\norder-of-magnitude ladder (illustrative):")
for label, mult in [
    ("1 call", 1),
    ("x10 agent run", 10),
    ("x100 dev loop", 100),
    ("x1000 deploy", 1000),
]:
    print(f"  {label:16s} ~ ${one_call * mult:.2f}")

turn 1: input re-sent= 200 tokens  call=$0.00150
turn 2: input re-sent= 400 tokens  call=$0.00210
turn 3: input re-sent= 600 tokens  call=$0.00270
turn 4: input re-sent= 800 tokens  call=$0.00330
turn 5: input re-sent=1000 tokens  call=$0.00390

5-turn session ~ $0.0135

order-of-magnitude ladder (illustrative):
  1 call           ~ $0.01
  x10 agent run    ~ $0.14
  x100 dev loop    ~ $1.35
  x1000 deploy     ~ $13.50


⚠️ **Pricing note:** the rates here are illustrative. Per-token pricing changes — always pull the
current Claude Sonnet numbers from Anthropic's pricing page before quoting a real figure.

The bow: everything you front-loaded to predict better is overhead you re-send, re-count, and
re-pay on every call — against both budgets. *Every prompt is a budget decision.*

[↑ Back to top](#top)

## 5. Common pitfalls: L01's four gotchas

L01's failure modes were woven *through* the demos, not saved for the end — so name them now as
portable pitfalls you can catch yourself committing later. One root ties them together: **tokens are
the unit of both budgets (the window *and* the bill), and there is no memory** — so every one of
these is the same discipline, *measure, don't guess.*

| Gotcha | Cure | Where you saw it |
| --- | --- | --- |
| **"A word is a token."** Eyeballing text as roughly word-count. | Count with the tokenizer before you budget — the ≈4-chars-per-token rule is *English-prose-only*; proper names fragment 3–5× and JSON / non-Latin blow past it. | Demo 3 (tokenization) |
| **"Temperature is just a randomness knob"** — and its cousin, *"temperature 0 is deterministic."* | Pick temperature by task (~0 for extraction / routing, ~1 for brainstorming); temp 0 is **low variance, not zero**, so never *promise* reproducibility from it alone. | Demo 3.6 (sampling) |
| **"Cost ≈ how long my prompt is."** Budgeting on input length and forgetting the reply. | Estimate *both* directions — output tokens cost ~3–5× input per token (§3 above). | §3 of this demo |
| **Silent context-window overflow / truncation.** Assuming an over-long request always errors loudly. | Track cumulative tokens against the window yourself — some frameworks **silently drop** the oldest turns, and quality degrades near the long-context tail *before* any hard limit. | §2 of this demo |

Two of the four are another lesson's whole topic — say the forward link and **don't re-teach it
here**: reproducibility (gotcha 2) returns as an eval gotcha in **L13**; silent overflow / truncation
(gotcha 4) is the whole of **L19** (context management) and part of **L21** (RAG), both full-course.

[↑ Back to top](#top)